In [ ]:
import pandas as pd
import numpy as np

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from src.data_loader import load_local_features

df = load_local_features()
print(df.shape)
print(df.head()) 

ModuleNotFoundError: No module named 'src'

In [2]:
def build_similarity_matrix(features, method='cosine'):
    """
    根据指定方法构建相似度矩阵
    method:
        - 'cosine'
        - 'euclidean'
    """
    if method == 'cosine':
        sim_matrix = cosine_similarity(features)

    elif method == 'euclidean':
        # 欧式距离越小越相似，所以要转成 similarity
        dist_matrix = euclidean_distances(features)

        # 转成相似度，值越大越相似
        sim_matrix = 1 / (1 + dist_matrix)

    else:
        raise ValueError("method 必须是 'cosine' 或 'euclidean'")

    return sim_matrix

In [3]:
# 1. 假设你已经把 BigQuery 的数据读取到了 df 中
# df = client.query(query).to_dataframe()

# 2. 选择用于计算“风格”的特征列
feature_cols = [
    'shots_p90', 'xg_p90', 'key_passes_p90', 'pass_accuracy',
    'carries_p90', 'dribble_success_rate', 'interceptions_p90',
    'duel_win_rate', 'recoveries_p90', 'mean_x', 'mean_y', 'activity_spread'
]

# 先复制，避免改坏原表
df_model = df.copy()

# 1. 缺失值填充（保持你原来的逻辑）
df_model[feature_cols] = df_model[feature_cols].fillna(0)

# 2. 按 position_group 做 z-score normalization
df_model_scaled = df_model.copy()

for col in feature_cols:
    group_mean = df_model_scaled.groupby('position_group')[col].transform('mean')
    group_std = df_model_scaled.groupby('position_group')[col].transform('std')
    group_std = group_std.fillna(1).replace(0,1)

    df_model_scaled[col] = (df_model_scaled[col] - group_mean) / group_std

# 3. 取标准化后的特征矩阵
scaled_features = df_model_scaled[feature_cols].values

pca = PCA(n_components=0.9)
pca_features = pca.fit_transform(scaled_features)

print("原始维度:", scaled_features.shape[1])
print("PCA 后维度:", pca_features.shape[1])

# 4. 计算 cosine similarity
sim_matrix_cosine = build_similarity_matrix(scaled_features, method='cosine')
sim_matrix_euclidean = build_similarity_matrix(scaled_features, method='euclidean')
sim_matrix_pca = build_similarity_matrix(pca_features, method='cosine')

print("Cosine sim matrix shape:", sim_matrix_cosine.shape)
print("Euclidean sim matrix shape:", sim_matrix_euclidean.shape)
print("PCA sim matrix shape:", sim_matrix_pca.shape)

原始维度: 12
PCA 后维度: 8
Cosine sim matrix shape: (2184, 2184)
Euclidean sim matrix shape: (2184, 2184)
PCA sim matrix shape: (2184, 2184)


In [4]:
def recommend_players(player_id, top_k=5, same_position_only=True, method='cosine'):
    target_id = str(player_id)

    # 选择相似度矩阵
    if method == 'cosine':
        sim_matrix = sim_matrix_cosine
    elif method == 'euclidean':
        sim_matrix = sim_matrix_euclidean
    elif method == 'pca':
        sim_matrix = sim_matrix_pca
    else:
        return "method 只能是 'cosine'、'euclidean' 或 'pca'"

    # 查找目标球员
    filtered_df = df_model_scaled[df_model_scaled['player_key'].astype(str) == target_id]

    if filtered_df.empty:
        raise ValueError(f"找不到 ID 为 {target_id} 的球员")

    target_row = filtered_df.iloc[0]
    target_idx = filtered_df.index[0]
    target_position = target_row['position_group']

    scores = list(enumerate(sim_matrix[target_idx]))

    results = []
    for idx, score in scores:
        candidate = df_model_scaled.iloc[idx]

        # 排除自己
        if str(candidate['player_key']) == target_id:
            continue

        # 同位置过滤
        if same_position_only and candidate['position_group'] != target_position:
            continue

        results.append({
            'player_key': candidate['player_key'],
            'player_name': candidate['player_name'],
            'position_group': candidate['position_group'],
            'similarity_score': round(float(score), 4)
        })

    recommend_df = pd.DataFrame(results)
    recommend_df = recommend_df.sort_values(by='similarity_score', ascending=False).head(top_k).reset_index(drop=True)

    return recommend_df

In [5]:
cosine_result = recommend_players("564228", top_k=5, method='cosine')
euclidean_result = recommend_players("564228", top_k=5, method='euclidean')
pca_result = recommend_players("564228", top_k=5, method='pca')

print("=== Cosine ===")
print(cosine_result)

print("\n=== Euclidean ===")
print(euclidean_result)

print("\n=== PCA + Cosine ===")
print(recommend_players("564228", method='pca'))

=== Cosine ===
  player_key           player_name position_group  similarity_score
0     564416             Asish Rai             DF            0.8673
1     197545            Ander Capa             DF            0.8635
2     630478  Parag Satish Shrivas             DF            0.8354
3     559482   Naorem Roshan Singh             DF            0.8055
4      27306       Mathieu Debuchy             DF            0.8000

=== Euclidean ===
  player_key      player_name position_group  similarity_score
0     564416        Asish Rai             DF            0.2996
1     197545       Ander Capa             DF            0.2959
2      27306  Mathieu Debuchy             DF            0.2634
3      91251     Cuco Martina             DF            0.2620
4      94807   Ognjen Vranjes             DF            0.2603

=== PCA + Cosine ===
  player_key           player_name position_group  similarity_score
0     197545            Ander Capa             DF            0.8637
1     564416          

In [6]:
cosine_ids = set(recommend_players("564228", method='cosine')['player_key'].astype(str))
euclidean_ids = set(recommend_players("564228", method='euclidean')['player_key'].astype(str))
pca_ids = set(recommend_players("564228", method='pca')['player_key'].astype(str))

print("Cosine vs Euclidean overlap:", len(cosine_ids & euclidean_ids))
print("Cosine vs PCA overlap:", len(cosine_ids & pca_ids))
print("Euclidean vs PCA overlap:", len(euclidean_ids & pca_ids))

Cosine vs Euclidean overlap: 3
Cosine vs PCA overlap: 4
Euclidean vs PCA overlap: 3


可视化雷达图 (Radar Chart)
这是足球分析中最常用的图表。你可以画一张雷达图，重叠对比“目标球员”和“推荐球员”的五维数据，直观展示他们到底有多像。

In [7]:
import matplotlib.pyplot as plt
import numpy as np

def plot_radar_comparison(target_id, recommend_id):
    # 定义五个核心维度 [cite: 20]
    categories = ['Shooting', 'Passing', 'Progression', 'Defensive', 'Spatial']

    # 简单的加权聚合：将多个 p90 指标归类为五个大维度
    df_radar = df_model.copy()
    df_radar['Shooting'] = df_radar[['shots_p90', 'xg_p90']].mean(axis=1)
    df_radar['Passing'] = df_radar[['pass_accuracy', 'key_passes_p90']].mean(axis=1)
    df_radar['Progression'] = df_radar[['carries_p90', 'dribble_success_rate']].mean(axis=1)
    df_radar['Defensive'] = df_radar[['interceptions_p90', 'duel_win_rate', 'recoveries_p90']].mean(axis=1)
    df_radar['Spatial'] = df_radar[['mean_x', 'activity_spread']].mean(axis=1)

    # 归一化到 0-1 方便绘图
    radar_scaler = StandardScaler()
    df_radar[categories] = radar_scaler.fit_transform(df_radar[categories])

    def get_values(p_id):
        name = df_radar[df_radar['player_key'] == str(p_id)]['player_name'].values[0]
        values = df_radar[df_radar['player_key'] == str(p_id)][categories].values.flatten().tolist()
        return name, values + values[:1] # 闭合图形

    name_a, val_a = get_values(target_id)
    name_b, val_b = get_values(recommend_id)

    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.plot(angles, val_a, color='blue', label=name_a)
    ax.fill(angles, val_a, color='blue', alpha=0.2)
    ax.plot(angles, val_b, color='red', label=name_b)
    ax.fill(angles, val_b, color='red', alpha=0.2)

    ax.set_thetagrids(np.degrees(angles[:-1]), categories)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.show()

# 调用：对比 564228 和它最像的球员
first_recommend_id = recommend_df.iloc[0]['player_key']
plot_radar_comparison("564228", first_recommend_id)

NameError: name 'recommend_df' is not defined

# **Scenario B — Condition-based Recruitment**

In [8]:
df_rank = df_model_scaled.copy()

# 这里先用你现有特征构造五个维度分数
df_rank['attacking_score'] = (
    df_rank['xg_p90'] +
    df_rank['shots_p90'] +
    df_rank['key_passes_p90']
) / 3

df_rank['progression_score'] = (
    df_rank['pass_accuracy'] +
    df_rank['carries_p90'] +
    df_rank['dribble_success_rate']
) / 3

df_rank['defensive_score'] = (
    df_rank['interceptions_p90'] +
    df_rank['duel_win_rate'] +
    df_rank['recoveries_p90']
) / 3

# 你当前表里如果没有 fouls/cards，就先给 discipline 留一个占位逻辑
# 如果后面表里有 fouls_p90 / cards_p90，再替换
df_rank['discipline_score'] = 0

df_rank['spatial_score'] = (
    df_rank['mean_x'] +
    df_rank['mean_y'] +
    df_rank['activity_spread']
) / 3

In [9]:
def recruit_players(
    target_position,
    top_k=10,
    attacking_weight=0.25,
    progression_weight=0.25,
    defensive_weight=0.25,
    discipline_weight=0.0,
    spatial_weight=0.25,
    age_min=None,
    age_max=None,
    market_value_max=None
):
    """
    Scenario B: Condition-based Recruitment
    """

    candidates = df_rank.copy()

    # 1. hard filter: 位置
    candidates = candidates[candidates['position_group'] == target_position].copy()

    # 2. optional filters
    if age_min is not None and 'age' in candidates.columns:
        candidates = candidates[candidates['age'] >= age_min]

    if age_max is not None and 'age' in candidates.columns:
        candidates = candidates[candidates['age'] <= age_max]

    if market_value_max is not None and 'market_value' in candidates.columns:
        candidates = candidates[candidates['market_value'] <= market_value_max]

    if candidates.empty:
        return pd.DataFrame()

    # 3. weighted composite score
    candidates['recruitment_score'] = (
        attacking_weight * candidates['attacking_score'] +
        progression_weight * candidates['progression_score'] +
        defensive_weight * candidates['defensive_score'] +
        discipline_weight * candidates['discipline_score'] +
        spatial_weight * candidates['spatial_score']
    )

    # 4. 输出结果
    output_cols = ['player_key', 'player_name', 'position_group', 'recruitment_score']

    # 如果表里存在这些列，就一起展示
    for col in ['age', 'market_value']:
        if col in candidates.columns:
            output_cols.append(col)

    result = (
        candidates[output_cols]
        .sort_values(by='recruitment_score', ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    return result

In [10]:
recruit_result = recruit_players(
    target_position='MF',
    top_k=10,
    attacking_weight=0.3,
    progression_weight=0.4,
    defensive_weight=0.2,
    discipline_weight=0.0,
    spatial_weight=0.1
)

print(recruit_result)

  player_key            player_name position_group  recruitment_score
0      68290                 Neymar             MF           1.350561
1     122153             Paul Pogba             MF           1.088404
2      60444       Thiago Alcántara             MF           1.064936
3       7607                   Xavi             MF           1.047106
4      50057           Aaron Ramsey             MF           1.043320
5     123406  Daniel Alves Da Silva             MF           0.969311
6     401578      Exequiel Palacios             MF           0.962893
7      55215         Javier Pastore             MF           0.896652
8     234953           Ahmed Jahouh             MF           0.888160
9     258027         Renato Sanches             MF           0.836653


In [11]:
import os

os.makedirs("../artifacts/features", exist_ok=True)

df_model_scaled.to_parquet("../artifacts/features/processed_features.parquet", index=False)

print("✅ Saved processed features!")

✅ Saved processed features!
